# Files, paths and common formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-06-files-and-formats.ipynb)

Most pipelines start and end with files: CSV exports, JSON from an API, compressed logs, and folders of partitioned
output. This chapter covers how to open files safely, how to read and write CSV and JSON, how to handle compressed
files and how to build paths without string gluing. Every example writes to a temporary folder, so it leaves no files
behind.

## Paths with `pathlib`

`pathlib.Path` joins paths with `/`, works on every operating system and gives you names and suffixes without string
slicing. Create a temporary folder for the examples below.

In [ ]:
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="py-files-"))
landing = workdir / "landing" / "orders"
landing.mkdir(parents=True)

file_path = landing / "orders_2024-01-15.csv"
assert file_path.name == "orders_2024-01-15.csv"
assert file_path.stem == "orders_2024-01-15"
assert file_path.suffix == ".csv"
assert file_path.parent == landing

## Reading and writing text safely

Always open files with `with`, so they close even when an error happens. Pass `encoding="utf-8"` so results do not
depend on the machine. `newline=""` is needed for CSV so line endings are handled correctly.

In [ ]:
text_path = landing / "notes.txt"
with text_path.open("w", encoding="utf-8") as handle:
    handle.write("first line\n")
    handle.write("café\n")

with text_path.open(encoding="utf-8") as handle:
    lines = handle.read().splitlines()

assert lines == ["first line", "café"]
assert text_path.stat().st_size == len("first line\ncafé\n".encode("utf-8"))   # bytes, not characters

## CSV with the `csv` module

`DictReader` yields one dictionary per row, keyed by the header. Values are always strings: convert them yourself.
`DictWriter` writes dictionaries back out. Never build CSV by joining strings with commas, because quoted fields and
commas inside values break it.

In [ ]:
import csv

orders_path = landing / "orders.csv"
rows = [
    {"order_id": 1, "customer": "Ana, Ltd", "amount": 120.5},
    {"order_id": 2, "customer": "Ben", "amount": 80},
]
with orders_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["order_id", "customer", "amount"])
    writer.writeheader()
    writer.writerows(rows)

with orders_path.open(newline="", encoding="utf-8") as handle:
    loaded = [{**r, "order_id": int(r["order_id"]), "amount": float(r["amount"])} for r in csv.DictReader(handle)]

assert loaded[0]["customer"] == "Ana, Ltd"      # the comma survived the round trip
assert loaded[1]["amount"] == 80.0

## JSON

`json.loads` reads a string, `json.dumps` writes one, and `json.load` / `json.dump` work on files. JSON has no dates or
decimals: a `datetime` must be converted to a string first. Keys in JSON objects are always strings.

In [ ]:
import json
from datetime import date

event = {"id": 7, "day": date(2024, 1, 15).isoformat(), "tags": ["new", "mobile"]}
text = json.dumps(event, sort_keys=True)
assert text == '{"day": "2024-01-15", "id": 7, "tags": ["new", "mobile"]}'

back = json.loads(text)
assert back["tags"] == ["new", "mobile"]

# JSON Lines: one JSON object per line. The standard format for logs and event streams.
events_path = landing / "events.jsonl"
with events_path.open("w", encoding="utf-8") as handle:
    for n in range(3):
        handle.write(json.dumps({"n": n}) + "\n")

with events_path.open(encoding="utf-8") as handle:
    assert [json.loads(line)["n"] for line in handle] == [0, 1, 2]

## Large files: read line by line

`read()` loads the whole file into memory. Iterating over the file object reads one line at a time, so memory stays flat
even for a file far bigger than RAM. Combine it with a generator to process each line as it arrives.

In [ ]:
big_path = landing / "big.txt"
with big_path.open("w", encoding="utf-8") as handle:
    for n in range(100_000):
        handle.write(f"{n}\n")

total = 0
with big_path.open(encoding="utf-8") as handle:
    for line in handle:           # one line in memory at a time
        total += int(line)
assert total == sum(range(100_000))

## Compressed files

`gzip.open` reads and writes compressed text with the same interface as `open`. Compressed files are smaller on disk
and cheaper to move, and most engines read them natively. Use the `"rt"` mode to read text.

In [ ]:
import gzip

gz_path = landing / "events.jsonl.gz"
with gzip.open(gz_path, "wt", encoding="utf-8") as handle:
    handle.write('{"n": 1}\n{"n": 2}\n')

with gzip.open(gz_path, "rt", encoding="utf-8") as handle:
    values = [json.loads(line)["n"] for line in handle]
assert values == [1, 2]
assert gz_path.read_bytes()[:2] == b"\x1f\x8b"   # the gzip magic number

## Writing a file atomically

If a job crashes while writing, a half-written file can look like a finished one. Write to a temporary name in the same
folder, then `replace` it into place. The replace is atomic on one filesystem: readers see the old file or the new file,
never a mix.

In [ ]:
import os

def write_atomic(path, text):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        handle.write(text)
        handle.flush()
        os.fsync(handle.fileno())
    tmp.replace(path)

target = landing / "status.json"
write_atomic(target, '{"ok": true}')
assert json.loads(target.read_text(encoding="utf-8")) == {"ok": True}
assert not (landing / "status.json.tmp").exists()

## Practice questions

1. List every `.csv` file under a folder, recursively, sorted by name. Use `pathlib`, not `os.walk`.
2. Count the lines of a `.gz` file without loading it into memory.
3. Write a function that returns the value of a `key` from each line of a JSON Lines file, skipping blank lines.

## Answers

### Question 1

In [ ]:
def csv_files(folder):
    return sorted(Path(folder).rglob("*.csv"))

found = csv_files(landing)
assert [p.name for p in found] == ["orders.csv"]   # the only CSV written to landing above

### Question 2

In [ ]:
def count_gzip_lines(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return sum(1 for _ in handle)

assert count_gzip_lines(gz_path) == 2

### Question 3

In [ ]:
def values_for_key(jsonl_path, key):
    with Path(jsonl_path).open(encoding="utf-8") as handle:
        return [json.loads(line)[key] for line in handle if line.strip()]

blank_path = landing / "blank.jsonl"
blank_path.write_text('{"k": 1}\n\n{"k": 2}\n', encoding="utf-8")
assert values_for_key(blank_path, "k") == [1, 2]

## Interview notes

- **Always use `with`** and an explicit encoding. Most "works on my machine" bugs are encoding or newline bugs.
- **Stream large files.** Iterate, do not `read()`. Count memory, not just correctness.
- **Write atomically.** A temporary file plus a rename turns a crash mid-write into "no output", which a retry can fix.
- **Compare the formats.** CSV has no types, JSON has no dates, JSON Lines suits streams, Parquet (next topics) suits
  analytics. Pick the format for how the data is read later.